# MLM vs. $\chi^2$

Cíl je porovnat modifikovaný MLM fit se standardním fitem pomocí $\chi^2$. Máme data co mají "divočejší taily", a tak můžou mít i záporné obsahy binů.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

## Generování dat
Reálná měření vždy obsahují šum. Simulujeme jako, normální rozdělení (signál), k němu přičteme uniformní šum a odečteme data z jiného uniformního rozdělení (simulované měření pozadí). 

Konečná data pro fitování jsou: `hist_data = h1 + h2 - h3`
* **h1:** Gaussovská data signálu.
* **h2:** Uniformní data pozadí.
* **h3:** Uniformní data pozadí.

Odečtení pozadí $h3$ umožňuje vznik binů i se záporným obsahem (záporné $N_k$). Standardní Poisson v MLM modeluje pravděpodobnost naměření $N_k$ událostí jako $$p(N_k|D_k) = \frac{D_k^{N_k} e^{-D_k}}{N_k!}.$$ Nepořádek dělá faktoriál ($N_k!$), který není pro záporná čísla definován. Proto modifikovaná MLM.

In [4]:
# params
np.random.seed(67) # for repoducibility
N_events = 10000
true_x0, true_sigma = 5.0, 1.0
x_min, x_max, n_bins = 0, 10, 50

# data
data_gauss = np.random.normal(true_x0, true_sigma, N_events)
data_flat_add = np.random.uniform(x_min, x_max, N_events)
data_flat_sub = np.random.uniform(x_min, x_max, N_events)

# data to histograms
h1, bin_edges = np.histogram(data_gauss, bins=n_bins, range=(x_min, x_max))
h2, _ = np.histogram(data_flat_add, bins=n_bins, range=(x_min, x_max))
h3, _ = np.histogram(data_flat_sub, bins=n_bins, range=(x_min, x_max))

# bin centres and contents
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
hist_data = h1 + h2 - h3

# simulated error for chi2 fit
bin_errors = np.sqrt(h1 + h2 + h3)
valid_bins = bin_errors > 0 

## Modifikovaný (ne)binovaný MLM fit

 Kvůli problému s Poisonem převedeme binovaný MLM na nebinovaný. Místo počítání statistiky $N_k$ měření v daném binu, budeme předstírat, že jsme provedli $N_k$ nezávislých měření, která všechna padla přesně na střed binu ($x_k$).

Hustota pravděpodobnosti, že jedno měření dopadne do bodu $x_k$, je: $$p(x_k | x_0, \sigma) = A \cdot \exp\left(-\frac{(x_k - x_0)^2}{2\sigma^2}\right),$$ kde $A = \frac{1}{\sqrt{2\pi}\sigma}$.

Pokud bereme $N_k$ událostí jako nezávislé měření co padly do $x_k$ tak celková pravděpodobnost pro tento bin ($N_k$ měření do něj padlo) je $N_k$-tá mocnina pravděpodobnosti jednoho měření: $$L = \left( A \cdot \exp\left(-\frac{(x_k - x_0)^2}{2\sigma^2}\right) \right)^{N_k}.$$

Vynásobením všech $M$ binů máme celkovou věrohodnost. Produkt a mocnina jsou pomalé, proto zlogaritmujeme:
   $$ \ln L = \sum_{k=1}^{M} N_k \left( \ln A - \frac{(x_k - x_0)^2}{2\sigma^2} \right).$$

In [5]:
def negative_log_likelihood(params):
    x0, sigma = params
    if sigma <= 0: return np.inf # Sigma must be positive
    
    A = 1.0 / (np.sqrt(2 * np.pi) * sigma)
    log_terms = np.log(A) - ((bin_centers - x0)**2) / (2 * sigma**2)
    
    log_likelihood = -np.sum(hist_data * log_terms)
    return log_likelihood

## $\chi^2$ fit

Metoda $\chi^2$ je specifickým případem MLM. Místo toho, že se biny plní podle Poissona, předpokládá, že obsah $k$-tého binu, $N_k$, je náhodná veličina z Gaussova rozdělení. Rozdělení je centrováno na očekávané hodnotě, $D_k$, se známou nejistotou, $\sigma_k$.

Likelihood pro všech $M$ binů je součinem jejich jednotlivých Gaussovských pravděpodobností:
   $$ L = \prod_{k=1}^{M} \frac{1}{\sqrt{2\pi}\sigma_k} \exp\left(-\frac{(N_k - D_k)^2}{2\sigma_k^2}\right) $$
Zlogaritmujeme....$\ln L = \text{konst.} - \frac{1}{2} \sum_{k=1}^{M} \left( \frac{N_k - D_k}{\sigma_k} \right)^2$ 

Člen se sumou je označíme jako $\chi^2$. Protože $\ln L = \text{konst.} - \frac{1}{2} \chi^2$, maximalizace pravděpodobnosti vyžaduje minimalizaci $\chi^2$.

$$ \chi^2 = \sum_{k=1}^{M} \left(\frac{N_k - D_k}{\sigma_k}\right)^2 $$

In [ ]:
def chi_square(params):
    Amplitude, x0, sigma = params
    if sigma <= 0: return np.inf
    
    # Exoected value of bin based
    D_k = Amplitude * np.exp(-((bin_centers - x0)**2) / (2 * sigma**2))
    
    # Výpočet chi2 pouze pro platné biny, aby se zabránilo dělení nulou
    chi2_val = np.sum(((hist_data[valid_bins] - D_k[valid_bins]) / bin_errors[valid_bins])**2)
    return chi2_val